In [1]:
# verify cuda
import torch
torch.cuda.is_available()

False

In [2]:
#torch.cuda.empty_cache()

In [3]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from collections import Counter
import pickle

from LLMGen import length_stratified_split, make_seq2seq_pairs, reduce_input_block
from LLMGen import filter_cases_by_length, sample_few_shots
from LLMGen import assemble_prompt_from_fewshots, generate_traces_for_batch_hf, llm_results_to_eventlog, gt_add_col

from Evaluation import evaluate_comprehensive, evaluate_light
from ollama import Client

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline, GenerationConfig

For CPU, loading model

In [9]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

C:\Users\Florence\miniforge3\envs\llm\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Florence\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 327.48it/s, Materializing param=

**FOR GPU, `bitsandbytes` 4-bit quantization requires bitsandbytes. pip install -U bitsandbytes>=0.46.1`** 

**BitsAndBytesConfig not working on CPU, BitsAndBytes requires an NVIDIA GPU with CUDA**

In [ ]:
#model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
#model_id = "mistralai/Mistral-7B-Instruct-v0.3"
#model_id = "Qwen/Qwen2.5-3B-Instruct"
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    device_map="auto"
)
pipe.generation_config = GenerationConfig(
    max_length=None,
    do_sample=False
)

In [10]:
train_event = pd.read_csv("../output/helpdesk_train.csv")
test_event = pd.read_csv("../output/helpdesk_hold.csv")

In [11]:
# Rename all columns
def rename(event, timecol, sequence_id):
    event = event.copy()
    event[timecol] = pd.to_datetime(event[timecol])
    event = event.sort_values([sequence_id, timecol])
    event.columns = [c.replace(":", "_") for c in event.columns]
    return event

train_event = rename(train_event, "time:timestamp", "case:concept:name")
test_event = rename(test_event,  "time:timestamp", "case:concept:name")

In [12]:
# Reuse the previous stored pairs for the second model
with open("../output/helpdesk_train_seq2seq.pkl", "rb") as f:
    train_pairs  = pickle.load(f)

with open("../output/helpdesk_hold_seq2seq.pkl", "rb") as f:
    test_pairs  = pickle.load(f)

In [13]:
# helpdesk dataset
length_buckets = {
    "short":  (3, 5),   # 271 traces
    "medium": (5, 7),   # 154 traces
    "long":   (7, None) # 32 traces (7–11 merged)
}

In [14]:

for bucket_name, (min_len, max_len) in length_buckets.items():

    print(f"\n=== Bucket: {bucket_name} ({min_len}, {max_len}) ===")

    # Sample few-shot examples from training
    few_shots = sample_few_shots(train_pairs, k=3, min_len=min_len, max_len=max_len, max_expand=5)

    if len(few_shots) == 0:
        print("[SKIP] No few-shot examples available.")
        continue
        
    # Sample hold-out cases for generation
    selected_holdouts = filter_cases_by_length(test_pairs, min_len=min_len, max_len=max_len)

    if len(selected_holdouts) == 0:
        print("[SKIP] No holdout cases in this bucket.")
        continue
    

    selected_holdouts_inputs= [reduce_input_block(p) for p in selected_holdouts]

    results = generate_traces_for_batch_hf(selected_holdouts_inputs, few_shots, pipe, max_new_token_num=300)

    for r in results:
        r["length_bucket"] = bucket_name

    all_generated.extend(results)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Bucket: short (3, 5) ===


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

KeyboardInterrupt: 